# AAM Loader Playground

Interactive notebook for exploring the `src.data.load_data` utilities on the `tinyAAM` dataset.

Each section below focuses on one part of the data-loading pipeline:
- finding the available tracks on disk
- reading the AAM annotation files into pandas tables
- loading one audio track and converting it into chroma features
- aligning chord annotations to feature frames so the result is ready for model training

## Setup

This cell makes sure the repository root is on `sys.path`.

That matters because notebooks often run with `notebooks/` as the current working directory, while the loader code lives under `src/`.

In [ ]:
from pathlib import Path
import sys

# Start from the notebook's current working directory.
repo_root = Path.cwd()
# If Jupyter launched inside notebooks/, move up to the repo root so imports work.
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

# Make the repository importable so `from src...` works in the notebook.
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

repo_root

In [ ]:
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Import the high-level dataset wrapper and the lower-level helpers.
from src.data import (
    AAMDataset,
    beat_annotations_to_intervals,
    extract_chroma_features,
    frame_labels_from_intervals,
    frame_times,
    load_training_example,
)

## Index the dataset

Create an `AAMDataset` object that points at your local `tinyAAM` folder.

The `list_tracks()` call builds a small summary table showing which track ids are available and where their mix audio and annotation files live.

In [ ]:
# Point the dataset wrapper at your local tinyAAM folder.
dataset = AAMDataset(repo_root / "data/raw/tinyAAM")
# Build a summary table of the tracks currently available on disk.
tracks = dataset.list_tracks()
print(f"Found {len(tracks)} tracks")
tracks.head()

In [ ]:
# Pick a track id to inspect. Change this to try another song.
track_id = "0001"
# Load the three annotation tables for this track.
annotations = dataset.load_annotations(track_id)

print("beatinfo")
# beatinfo stores beat-level chord labels.
display(annotations["beatinfo"].head())

print("segments")
# segments stores larger structural regions and keys.
display(annotations["segments"].head())

This section is useful for checking that the parser is reading the AAM annotations correctly.

- `beatinfo` contains beat-level chord labels.
- `segments` contains larger structural sections and their keys.

If you want to inspect another song, just change `track_id` above.

## Load one training example

`load_training_example(...)` is the highest-level helper in `load_data.py`.

It performs the most common pipeline for training:
- load the mix audio
- compute chroma features
- compute a timestamp for each frame
- align the chord annotations to those frame timestamps

In [ ]:
# Run the full preprocessing path for one training example.
example = load_training_example(track_id, root=repo_root / "data/raw/tinyAAM")

print("audio shape:", example["audio"].shape)
print("sample rate:", example["sample_rate"])
print("chroma shape:", example["chroma"].shape)
print("frame labels shape:", example["frame_labels"].shape)
print("first 20 labels:", example["frame_labels"][:20].tolist())

In [ ]:
# Count how often each chord label appears in the aligned frame labels.
unique_labels = pd.Series(example["frame_labels"]).value_counts()
unique_labels.head(15)

This quick count shows which chord labels appear most often in the chosen track.

That is a simple sanity check that your label alignment is producing realistic chord names instead of empty strings or parsing artifacts.

## Plot the chroma features

This plot shows the chroma matrix that would typically be fed into a chord-recognition model.

The rows correspond to pitch classes and the columns correspond to time frames.

In [ ]:
# Visualize the chroma representation over time.
plt.figure(figsize=(14, 5))
librosa.display.specshow(
    example["chroma"],
    x_axis="time",
    y_axis="chroma",
    sr=example["sample_rate"],
    hop_length=512,
)
plt.colorbar()
plt.title(f"Chroma features for track {track_id}")
plt.tight_layout()
plt.show()

## Inspect frame labels against beat annotations

The next two cells make the time alignment more transparent.

First we convert beatwise chord annotations into explicit time intervals, then we inspect the first few frame timestamps and the labels assigned to them.

In [ ]:
# Convert beat timestamps into explicit [start, end) chord intervals.
intervals = beat_annotations_to_intervals(
    example["beatinfo"],
    audio_duration=len(example["audio"]) / example["sample_rate"],
)
intervals.head(12)

In [ ]:
# Show the first few frame timestamps and the chord label assigned to each one.
preview = pd.DataFrame(
    {
        "time": example["frame_times"][:40],
        "label": example["frame_labels"][:40],
    }
)
preview.head(20)

## Manual step-by-step version

Use this section if you want to swap parameters without calling `load_training_example`.

This is the best place to experiment with things like:
- a different `hop_length`
- loading a different `track_id`
- trying a different feature function later on

It mirrors the higher-level helper, but exposes each intermediate result directly.

In [ ]:
# Load the raw mix audio at a fixed sample rate.
audio, sr = dataset.load_audio(track_id, source="mix", sr=22050, mono=True)
# Pull just the beatwise chord annotations for manual alignment.
beatinfo = dataset.load_annotations(track_id)["beatinfo"]

# Larger hop_length means fewer time frames and coarser time resolution.
hop_length = 1024
# Compute chroma features from the raw waveform.
chroma = extract_chroma_features(audio, sr, hop_length=hop_length)
# Compute the timestamp of each chroma frame.
times = frame_times(chroma.shape[1], sr, hop_length=hop_length)
# Convert beat annotations into chord intervals spanning time.
intervals = beat_annotations_to_intervals(beatinfo, audio_duration=len(audio) / sr)
# Assign a chord label to every feature frame.
labels = frame_labels_from_intervals(intervals, times)

print("manual chroma shape:", chroma.shape)
print("manual labels shape:", labels.shape)
print("manual label sample:", labels[:20].tolist())